# Procesamiento Experimental (video l3.mp4)

Este cuaderno procesa el video del cohete y construye su trayectoria.

**Datos del ensayo:** botella de 1.5 L, fracción de agua φ = 1/3 (0.5 L), inflada con bombín de bicicleta (la presión no quedó registrada).

**Geometría del montaje:** lanzador de 1.80 m; cámara a 18 m de distancia, sobre un trípode de 1.71 m.

**Antes de ejecutar:** sube `l3.mp4` a la misma carpeta que este cuaderno (en Colab: panel de archivos de la izquierda).

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

g = 9.81  # gravedad (m/s2)

### 1.1 Lectura del video

In [ ]:
VIDEO = "l3.mp4"

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)
ANCHO = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ALTO = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Cuadros por segundo: {FPS:.3f}")
print(f"Resolucion: {ANCHO} x {ALTO} pixeles")


def obtener_frame(t):
    """Devuelve el cuadro del video en el instante t (s) como imagen RGB."""
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, round(t * FPS))
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"No se pudo leer el video en t = {t} s")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

### 1.1 Explorador de cuadros

Revisión cuadro por cuadro.

In [ ]:
def ver_cuadros(t_inicio, t_fin, paso=0.1, columnas=5, con_grilla=True):
    """Muestra en una grilla los cuadros entre t_inicio y t_fin, uno cada 'paso' segundos."""
    tiempos = np.arange(t_inicio, t_fin, paso)
    filas = int(np.ceil(len(tiempos) / columnas))
    fig, axs = plt.subplots(filas, columnas, figsize=(3 * columnas, 5.3 * filas))
    axs = np.array(axs).reshape(-1)

    for ax, t in zip(axs, tiempos):
        ax.imshow(obtener_frame(t))
        if con_grilla:
            for x in range(0, ANCHO, 100):
                ax.axvline(x, color="red", lw=0.3, alpha=0.5)
            for y in range(0, ALTO, 100):
                ax.axhline(y, color="lime", lw=0.3, alpha=0.5)
        ax.set_title(f"t = {t:.3f} s", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axs[len(tiempos):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

ver_cuadros(19.7, 21, paso=1 / FPS)

### 1.2 Calibración de escala: altura de una persona

El operador, de pie a los t = 3.0 s, mide 1.8 m.

In [ ]:
t_persona = 3.0
frame_persona = obtener_frame(t_persona)

cabeza_px = 555   # fila del tope del casco
pies_px = 638     # fila donde el zapato toca el pasto
altura_px = pies_px - cabeza_px
altura_m = 1.8

escala = altura_m / altura_px   # metros por pixel
print(f"Altura de la persona: {altura_px} px")
print(f"Escala: {escala:.5f} m/px")

fig, ax = plt.subplots(figsize=(3.5, 5))
ax.imshow(frame_persona)
ax.axhline(cabeza_px, color="red", lw=1.5, label="cabeza")
ax.axhline(pies_px, color="lime", lw=1.5, label="pies")
ax.set_xlim(150, 200)
ax.set_ylim(650, 520)
ax.legend(fontsize=8, loc="upper right")
ax.set_title(f"Calibracion de escala (t = {t_persona} s)\naltura = {altura_px} px")
plt.tight_layout()
plt.show()

### 1.3 Posiciones del cohete (medidas a mano y detección de control)

In [ ]:
GANANCIA = 2.5   # video nocturno: se sube el brillo (tambien en la verificacion visual)


def candidatos_blancos(frame_rgb, s_max=60, v_min=175):
    """Manchas blancas o brillantes y compactas del cuadro.

    Devuelve una lista de (columna, fila, area).
    """
    brillante = np.clip(frame_rgb.astype(float) * GANANCIA, 0, 255).astype(np.uint8)
    hsv = cv2.cvtColor(brillante, cv2.COLOR_RGB2HSV)
    mascara = ((hsv[:, :, 1] < s_max) & (hsv[:, :, 2] >= v_min)).astype(np.uint8) * 255

    n_manchas, _, stats, centroides = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    candidatos = []
    for k in range(1, n_manchas):
        x, y, w, h, area = stats[k]
        if area < 4:
            continue
        if area / (w * h) < 0.15:   # descarta manchas muy ralas
            continue
        candidatos.append((centroides[k][0], centroides[k][1], area))
    return candidatos


def trackear(cuadros, semilla_xy, semilla_v, s_max=60, v_min=175):
    """Sigue el cohete cuadro a cuadro: elige la mancha mas cercana a la posicion predicha.

    semilla_xy: (columna, fila) en el primer cuadro (px); semilla_v: desplazamiento por cuadro (px).
    Si un cuadro no tiene candidatos, se mantiene la posicion predicha.
    Devuelve (columnas, filas, cuadros sin deteccion).
    """
    x, y = semilla_xy
    vx, vy = semilla_v
    columnas, filas, sin_deteccion = [x], [y], 0

    for k in cuadros[1:]:
        candidatos = candidatos_blancos(obtener_frame(k / FPS), s_max, v_min)
        x_pred, y_pred = x + vx, y + vy
        if candidatos:
            mejor = min(candidatos, key=lambda c: (c[0] - x_pred) ** 2 + (c[1] - y_pred) ** 2)
            x_nuevo, y_nuevo = mejor[0], mejor[1]
        else:
            x_nuevo, y_nuevo = x_pred, y_pred
            sin_deteccion += 1
        vx, vy = x_nuevo - x, y_nuevo - y
        x, y = x_nuevo, y_nuevo
        columnas.append(x)
        filas.append(y)

    return np.array(columnas), np.array(filas), sin_deteccion

In [ ]:
def interp_extrap(k, kd, v, n_fit=3):
    """Interpolacion lineal; despues del ultimo dato usa una recta ajustada a los ultimos n_fit puntos."""
    out = np.interp(k, kd, v)
    fuera = k > kd[-1]
    if fuera.any():
        p = np.polyfit(kd[-n_fit:], v[-n_fit:], 1)
        out[fuera] = np.polyval(p, k[fuera])
    return out


# (cuadro, columna_px, fila_px, dx_camara_px, dy_camara_px)
# dy_camara: cuanto bajo la escena respecto al cuadro de calibracion (la camara estaba en la mano)
datos = [
    (472, 137.5, 508.4, 9.2, -19.5),
    (473, 128.9, 470.0, 9.2, -19.5),
    (474, 113.0, 434.0, 9.1, -19.7),   # interpolado (casi invisible en este cuadro)
    (475,  96.7, 396.9, 8.9, -19.9),
    (476,  88.8, 376.4, 8.9, -20.0),
    # cuadros 477-480 y 482: el cohete no se distingue
    (481,  45.7, 296.5, 8.8, -21.0),
    (483,  32.0, 283.3, 8.7, -21.1),
    (484,  26.9, 280.2, 8.6, -21.5),
    (485,  21.3, 272.8, 8.6, -21.7),
    (486,  20.8, 272.0, 8.6, -21.9),   # dudoso: casi igual al 485
    (487,   9.8, 267.3, 8.4, -22.0),
    (488,   5.8, 265.4, 8.4, -22.2),
]

D = np.array(datos, float)
k_dat = D[:, 0]
estimados = [474]                        # puntos de la tabla que no son medicion directa

cuadros = np.arange(472, 489 + 1)        # el ultimo cuadro (489, ~20.5 s) se extrapola
t_visibles = cuadros / FPS

col_px = interp_extrap(cuadros, k_dat, D[:, 1])
fila_px = interp_extrap(cuadros, k_dat, D[:, 2])
dy_cam = interp_extrap(cuadros, k_dat, D[:, 4])

# True = medido; False = interpolado, estimado o extrapolado
medido = np.isin(cuadros, k_dat) & ~np.isin(cuadros, estimados)

In [ ]:
# Detector aplicado a los cuadros desde el primer al ultimo dato de la tabla.
# La posicion y el desplazamiento iniciales salen de los dos primeros puntos.
def imprimir_datos(cuadros, col_px, fila_px, dx_cam, dy_cam, por_linea=2):
    "Imprime las posiciones en el formato de la lista 'datos'."
    filas = [f"({k}, {c:5.1f}, {f:5.1f}, {dx:5.1f}, {dy:5.1f})" for k, c, f, dx, dy in zip(cuadros, col_px, fila_px, dx_cam, dy_cam)]
    print("datos = [")
    for i in range(0, len(filas), por_linea):
        print("    " + ", ".join(filas[i:i + por_linea]) + ",")
    print("]")


cuadros_det = cuadros[cuadros <= k_dat[-1]]
col_det, fila_det, sin_det = trackear(cuadros_det, (col_px[0], fila_px[0]),
                                      (col_px[1] - col_px[0], fila_px[1] - fila_px[0]))

en_tabla = np.isin(cuadros_det, k_dat)   # se compara solo en los cuadros de la tabla
imprimir_datos(cuadros_det[en_tabla], col_det[en_tabla], fila_det[en_tabla], D[:, 3], D[:, 4])
print(f"\nDiferencia maxima con 'datos': {np.abs(col_det[en_tabla] - D[:, 1]).max():.1f} px en columna, "
      f"{np.abs(fila_det[en_tabla] - D[:, 2]).max():.1f} px en fila")
print(f"Cuadros sin deteccion: {sin_det} de {len(cuadros_det)}")

**Verificación visual.**

In [ ]:
R = 30       # semilado de la ventana (px)
P = 2 * R    # relleno: permite puntos hasta R px fuera de la imagen
ncols = 6
nrows = int(np.ceil(len(cuadros) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.6 * nrows))
axs = np.atleast_1d(axs).reshape(-1)

for ax, k, t, xc, yc, ok in zip(axs, cuadros, t_visibles, col_px, fila_px, medido):
    padded = np.pad(obtener_frame(t), ((P, P), (P, P), (0, 0)), mode="constant")
    x0, y0 = int(round(xc)), int(round(yc))
    recorte = padded[y0 - R + P:y0 + R + P, x0 - R + P:x0 + R + P]
    recorte = np.clip(recorte.astype(float) * GANANCIA, 0, 255).astype(np.uint8)   # video nocturno: se sube el brillo
    ax.imshow(recorte, extent=[x0 - R, x0 + R, y0 + R, y0 - R])
    ax.plot(xc, yc, "o", mfc="none", mec="red" if ok else "yellow", ms=14, mew=2)   # rojo = medido, amarillo = no medido
    ax.set_title(f"k = {k}  t = {t:.3f} s" + ("" if ok else " (interp.)"), fontsize=8)
    ax.axis("off")
for ax in axs[len(cuadros):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1.4 Pasando a unidades físicas

In [ ]:
t_rel = t_visibles - t_visibles[0]
n = len(t_rel)

# Altura con escala constante (aproximacion), corrigiendo el movimiento de la camara
y_lineal = (pies_px - (fila_px - dy_cam)) * escala

# Altura con correccion de perspectiva (camara a D_cam m de distancia, sobre tripode de h_cam m)
D_cam, h_cam = 18.0, 1.71                 # (m)
f_px = D_cam / escala                    # distancia focal equivalente (px)
cy = ALTO / 2                            # fila central de la imagen
inclinacion0 = np.arctan((pies_px - cy) / f_px) - np.arctan(h_cam / D_cam)   # inclinacion de la camara en la calibracion
elevacion = inclinacion0 + np.arctan((cy - (fila_px - dy_cam)) / f_px)
y_m = h_cam + D_cam * np.tan(elevacion)  # altura sobre el suelo (m)
x_m = col_px * escala

print(f"Inclinacion de la camara en la calibracion: {np.degrees(inclinacion0):.1f} grados")
print(f"Distancia focal equivalente: {f_px:.0f} px\n")


def derivada(t, x):
    """Velocidad por diferencias centrales (en los extremos, diferencias simples)."""
    v = np.zeros(len(t))
    v[1:-1] = (x[2:] - x[:-2]) / (t[2:] - t[:-2])
    v[0] = (x[1] - x[0]) / (t[1] - t[0])
    v[-1] = (x[-1] - x[-2]) / (t[-1] - t[-2])
    return v


vx = derivada(t_rel, x_m)
vy = derivada(t_rel, y_m)

# Aceleracion vertical por segunda diferencia (sin valor en los extremos)
ay = np.full(n, np.nan)
ay[1:-1] = (y_m[2:] - 2 * y_m[1:-1] + y_m[:-2]) / (1 / FPS) ** 2

print(pd.DataFrame({"t (s)": t_rel.round(3), "x (m)": x_m.round(2), "y (m)": y_m.round(2),
                    "vx (m/s)": vx.round(2), "vy (m/s)": vy.round(2), "ay (m/s2)": ay.round(1)}).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(t_rel, y_m, "o-")
ax[0].set_ylabel("y (m)")
ax[0].set_title("Altura")
ax[1].plot(t_rel, vy, "o-")
ax[1].set_ylabel("vy (m/s)")
ax[1].set_title("Velocidad vertical")
for a in ax:
    a.set_xlabel("t (s)")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 1.4b Estimación del tramo fuera de cuadro (movimiento parabólico)

El apogeo no se ve en el video (el cohete sale por el borde superior). Se ajusta una parábola `y(t) = y0 + v0·t + a·t²/2` a los puntos medidos para estimar el apogeo.

- **Modelo A:** parábola libre (la aceleración sale del ajuste e incluye, de forma aproximada, el arrastre).
- **Modelo B:** parábola ideal con g = 9.81 m/s² fijo.

Si la gravedad efectiva del modelo A se aleja mucho de 9.81, los datos no calzan con una parábola ideal: conviene tomar el apogeo como un **rango**.

In [ ]:
def ajustar_parabola(t, y, g_fijo=None):
    """Devuelve (y0, v0, a) de y(t) = y0 + v0*t + a*t^2/2."""
    if g_fijo is None:
        c2, c1, c0 = np.polyfit(t, y, 2)
        return c0, c1, 2 * c2
    A = np.c_[np.ones_like(t), t]
    (y0, v0), *_ = np.linalg.lstsq(A, y + 0.5 * g_fijo * t ** 2, rcond=None)
    return y0, v0, -g_fijo


def altura(p, t):
    return p[0] + p[1] * t + 0.5 * p[2] * t ** 2


def velocidad(p, t):
    return p[1] + p[2] * t


def tiempos_fuera_de_cuadro(p):
    """Instantes en que la parabola cruza el borde superior del cuadro (o None si no lo cruza)."""
    y0, v0, a = p
    disc = v0 ** 2 - 2 * a * (y0 - y_borde)
    if disc < 0:
        return None
    return sorted([(-v0 + s * np.sqrt(disc)) / a for s in (1, -1)])


# Altura del borde superior del cuadro (fila 0), con la inclinacion media de la camara
elev_borde = inclinacion0 + np.arctan((cy + dy_cam.mean()) / f_px)
y_borde = h_cam + D_cam * np.tan(elev_borde)

modelos = [
    ("A: parabola libre (mejor ajuste)", ajustar_parabola(t_rel, y_m), "tab:red", "-"),
    ("B: parabola ideal, g = 9.81 m/s2", ajustar_parabola(t_rel, y_m, g_fijo=g), "tab:blue", "--"),
]

print(f"Borde superior del cuadro: {y_borde:.1f} m sobre el suelo\n")
apogeos = []
for nombre, p, _, _ in modelos:
    t_ap = -p[1] / p[2]
    y_ap = altura(p, t_ap)
    rms = np.sqrt(np.mean((y_m - altura(p, t_rel)) ** 2))
    apogeos.append((y_ap, t_ap))
    print(f"Modelo {nombre}")
    print(f"   Apogeo: {y_ap:.2f} m sobre el suelo, a t = {t_ap:.2f} s (t = 0 es el primer punto medido)")
    print(f"   Gravedad efectiva: {-p[2]:.1f} m/s2")
    print(f"   Velocidad de subida en t = 0: {p[1]:.1f} m/s")
    print(f"   Error RMS del ajuste: {rms:.2f} m")
    fuera = tiempos_fuera_de_cuadro(p)
    if fuera is not None and y_ap > y_borde:
        t1, t2 = fuera
        print(f"   Fuera de cuadro (estimado): de {t1:.2f} s a {t2:.2f} s ({t2 - t1:.2f} s)")
        print(f"   Velocidad al salir {velocidad(p, t1):+.1f} m/s, en el apogeo 0.0 m/s, al volver {velocidad(p, t2):+.1f} m/s")
    print()

ya = [a[0] for a in apogeos]
print(f"Resumen: apogeo estimado entre {min(ya):.1f} y {max(ya):.1f} m sobre el suelo "
      f"(mejor ajuste, modelo A: {ya[0]:.1f} m).")

# Graficos
t_plot = np.linspace(t_rel[0] - 0.15, t_rel[-1] + 0.1, 400)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(t_rel, y_m, "ko", ms=5, label="medido")
ax[1].plot(t_rel, vy, "ko", ms=5, label="medido")

for (nombre, p, color, estilo), (y_ap, t_ap) in zip(modelos, apogeos):
    ax[0].plot(t_plot, altura(p, t_plot), estilo, color=color, label=nombre)
    ax[1].plot(t_plot, velocidad(p, t_plot), estilo, color=color, label=nombre)
    ax[0].plot(t_ap, y_ap, "*", color=color, ms=14)
    ax[0].annotate(f"apogeo {y_ap:.1f} m", (t_ap, y_ap), textcoords="offset points",
                   xytext=(8, 6 if estilo == "-" else -14), color=color, fontsize=9)

ax[0].set_ylim(top=max(ya) + 1.5)
ax[0].axhline(y_borde, color="gray", ls=":", lw=1)
ax[0].text(t_plot[0], y_borde + 0.15, "borde superior del cuadro", color="gray", fontsize=8)
fuera_A = tiempos_fuera_de_cuadro(modelos[0][1])
if fuera_A is not None:
    ax[0].axvspan(*fuera_A, color="gray", alpha=0.15, label="fuera de cuadro (modelo A)")
    ax[1].axvspan(*fuera_A, color="gray", alpha=0.15)
ax[0].set_ylabel("altura sobre el suelo (m)")
ax[0].set_title("Altura: medido y ajuste parabolico")
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].set_ylabel("vy (m/s)")
ax[1].set_title("Velocidad vertical")
for a in ax:
    a.set_xlabel("t (s)")
    a.grid(alpha=0.3)
    a.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 1.5 Exportar a Excel

In [ ]:
df_exp = pd.DataFrame({
    "t": np.round(t_rel, 4),
    "y": np.round(y_m, 3),
    "y_lineal": np.round(y_lineal, 3),
    "vy": np.round(vy, 2),
    "ay": np.round(ay, 1),
    "step": np.arange(n),
    "tramo": 1,
})
df_exp.to_excel("Dato_exp_l3.xlsx", index=False)
df_exp